# BDA Competition - Final Ensemble

## submission_10files_7agree.csv 생성

10개 앙상블 파일을 로드하여 7개 이상 동의 시 1 예측

### 10개 파일 구성
| # | 파일명 | 설명 |
|---|--------|------|
| 1 | meta_vote_both | 5models_4agree AND enhanced_3agree |
| 2 | enhanced_3agree | Enhanced 모델 3개 이상 동의 |
| 3 | simcse_bert_4agree | SimCSE BERT 기반 |
| 4 | mega_ensemble_3agree | 메가 앙상블 |
| 5 | top3_2agree | Top 3 모델 2개 이상 동의 |
| 6 | prob_avg_035 | 확률 평균 threshold 0.35 |
| 7 | 10models_7agree | 10개 모델 7개 이상 동의 |
| 8 | 5models_4agree | 5개 모델 4개 이상 동의 |
| 9 | bert_data | BERT 단일 모델 |
| 10 | 5models_3agree | 5개 모델 3개 이상 동의 |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# 경로 설정
MODELS_DIR = Path("models")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Models Directory: {MODELS_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

## 1. 10개 모델 파일 로드

In [ ]:
# 10개 파일 목록
file_names = [
    'submission_meta_vote_both.csv',
    'submission_enhanced_3agree.csv',
    'submission_simcse_bert_4agree.csv',
    'submission_mega_ensemble_3agree.csv',
    'submission_top3_2agree.csv',
    'submission_prob_avg_035.csv',
    'submission_10models_7agree.csv',
    'submission_5models_4agree.csv',
    'submission_bert_data.csv',
    'submission_5models_3agree.csv',
]

# 파일 로드
preds = {}
test_ids = None

print("=" * 60)
print("10개 모델 파일 로드")
print("=" * 60)

for filename in file_names:
    filepath = MODELS_DIR / filename
    if filepath.exists():
        df = pd.read_csv(filepath)
        name = filename.replace('submission_', '').replace('.csv', '')
        preds[name] = df['completed'].values
        if test_ids is None:
            test_ids = df['ID'].values
        print(f"  {name}: {df['completed'].sum()} positives ({df['completed'].mean()*100:.1f}%)")
    else:
        print(f"  [ERROR] {filename} not found!")

print(f"\nTotal files loaded: {len(preds)}/10")

## 2. 앙상블: 7개 이상 동의 시 1 예측

In [ ]:
# 투표 합계 계산
vote_sum = np.zeros(len(test_ids))
for name, pred in preds.items():
    vote_sum += pred

# 투표 분포 확인
print("=" * 60)
print("투표 분포")
print("=" * 60)
for v in range(11):
    count = (vote_sum == v).sum()
    if count > 0:
        print(f"  {v}개 동의: {count}개 ({count/len(test_ids)*100:.1f}%)")

In [ ]:
# 7개 이상 동의 시 1
THRESHOLD = 7
final_pred = (vote_sum >= THRESHOLD).astype(int)

print("=" * 60)
print(f"최종 앙상블 (>= {THRESHOLD} 동의)")
print("=" * 60)
print(f"  Positives: {final_pred.sum()}")
print(f"  Negative: {(final_pred == 0).sum()}")
print(f"  Positive Rate: {final_pred.mean()*100:.2f}%")

## 3. 제출 파일 저장

In [ ]:
# 제출 파일 생성
submission = pd.DataFrame({
    'ID': test_ids,
    'completed': final_pred
})

# 저장
output_path = OUTPUT_DIR / "submission_10files_7agree.csv"
submission.to_csv(output_path, index=False)

print("=" * 60)
print("FINAL SUBMISSION")
print("=" * 60)
print(f"  File: {output_path}")
print(f"  Total samples: {len(submission)}")
print(f"  Positives: {submission['completed'].sum()}")
print(f"  Positive Rate: {submission['completed'].mean()*100:.2f}%")
print("\nReady for submission!")

In [ ]:
# 결과 미리보기
print("\n첫 10개 예측:")
print(submission.head(10))